# 第一轮：用普通 Python 完成一次实验

这是独立的合成教学项目，不连接设备。首次先完成第 0–4 题，再用 `reopen.ipynb` 验证新 kernel 重开并填写观察；第 5–7 题是后续可选练习。再次练习时，只重跑改参数、扫描、采集的相关单元；再次保存需换一个新版本名。最后的关闭连接是可选收尾，准备继续修改时不要关闭。
每题填写自己的观察；提示在 `HINTS.md`，参考解在 `REFERENCE.md`，真人体验记录在 `OBSERVATION.md`。

## 0. 打开已安装的环境
选择项目 `.venv` 的 Python 内核。下面已经给出连接和教学起点，不需要自己声明配置。

In [ ]:
import sys
from pathlib import Path

try:
    import scopecat as sc
except ModuleNotFoundError as error:
    raise RuntimeError(
        "请先运行 VS Code 任务“首次准备项目环境”, "
        "再在 Select Kernel 中选择本项目 .venv。"
    ) from error

project = sc.open_project()
if Path(sys.prefix).resolve() != (project.root / ".venv").resolve():
    raise RuntimeError(
        f"当前内核是 {sys.executable}; "
        f"请在 Select Kernel 中选择 {project.root / '.venv'}。"
    )

In [ ]:
import json
from pathlib import Path

import numpy as np

import scopecat as sc
from lab_teaching.parameters import Drive
from lab_teaching.session import analyze_rabi, open_parameters

project = sc.open_project()
session = project.authoring()
session.refresh()
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)
drive = params[Drive]
print(drive["q0"])

## 1. 查看和修改一个参数
`frequency` 的单位是 GHz。输入 `drive["q0"].` 可以查看属性补全；这一轮直接使用已经安装好的 `Drive`。

**动手：** 将下面频率改为 `5.148`，运行单元。预期看到频率的修改差异。
保存之前的修改也可以用于预览；它们不会自动发布为实验室默认值。

In [ ]:
drive["q0"].frequency = 5.15  # 动手：改为 5.148
print(params.diff())
_ = params.preview()
print("参数检查通过")

观察：我改动了 ______；单位是 ______；差异中的旧值是 ______。

## 2. 改扫描并预览
`np.linspace(起点, 终点, 点数)` 生成扫描值，幅度单位是 arb。
**动手：** 将点数改为 `21`，检查预览点数。暂时保留覆盖 `0～0.8` 的范围。
预览不会采集。每次改变参数或扫描后，都重新运行这个单元。
`teaching_rabi(...)` 创建本次实验请求；`request.values` 可以编辑，`sc.Scan(...)` 明确表示扫描。`prepare` 捕获当前请求和参数，之后的修改要重新预览才会生效。


In [ ]:
amplitudes = np.linspace(0, 0.8, 41)  # 动手：把 41 改为 21
request = teaching_rabi(shots=64, seed=200)
request.values["amplitude"] = sc.Scan(amplitudes)
prepared = session.prepare(request, parameters=params)
print("预览点数：", prepared.preview.point_count)

## 3. 采集、看图和读取已有分析
下面调用已提供的注册分析，第一轮不需要编写拟合器。一次 `run()` 是一次新采集；`wait()` 等待已有任务。
预期看到振荡响应和 `passed`，拟合候选接近 `0.24 arb`。这是教学模型的候选，不会写回参数表。

In [ ]:
job = prepared.run()
print("任务 receipt：", job.receipt)
run = job.wait(timeout=120).result()
report = analyze_rabi(session, run)
print(report.status, report.pi_amplitude, report.message)
report.figure

观察：图上大约有 ______ 个峰；候选为 ______；它是否已经成为默认参数？______。

## 4. 保存参数，重开数据
使用自己的版本名；已有版本名不能覆盖。打印并记下 `version.name` 和 `run.id`。
重开只读取已有结果，不会再次采集。

In [ ]:
version = params.save("my-first-drive", note="第一轮教学练习")  # 再次保存时改用新名字
print("参数版本：", version.name, "运行：", run.id)
reopened = session.config.workspace(context=version.name)
print(reopened[Drive]["q0"])
old_run = session.run(run.id)
print("重开已有运行：", old_run.id)
bookmark = {
    "parameter_version": version.name,
    "run_id": run.id,
    "receipt": str(job.receipt),
}
_ = (project.root / "notebooks/first-run.json").write_text(
    json.dumps(bookmark), encoding="utf-8"
)

关闭或重启 kernel 后，打开同目录的 `reopen.ipynb`，选择项目 `.venv` 的 Python 内核 并执行。它从 `first-run.json` 读取保存的版本与 receipt，重新绑定 `params`、`drive`，只恢复已有任务和数据，不保存、不提交采集。

第一次 `wait()` 超时时，当前 kernel 仍可继续 `job.wait(timeout=120).result()`；不要重跑 `prepared.run()`。其他任务也保留各自打印的 receipt，可用 `session.reopen(Path("实际 receipt 路径"))` 恢复。

首轮可以在这里暂停，先填写 `OBSERVATION.md`。如果本 Notebook 的内核仍在，可以直接继续。若已重启内核，先执行 HINTS.md 的“暂停后继续”代码恢复参数和结果；不要为恢复结果再次运行第 3 题。

## 5. 可选：失败也应当能读懂
下面复制原请求，只修改副本的扫描范围与 seed；原请求保持不变。故意只扫很窄的一段。预期图上看不到充分振荡，不产生候选（状态可能为 `needs_attention` 或 `fit_failed`）。
**动手：** 阅读 `message`，扩大范围后重新采集；不要把失败候选替换为参考答案。


In [ ]:
narrow_request = request.copy()
narrow_request.values["amplitude"] = sc.Scan(np.linspace(0, 0.1, 11))
narrow_request.values["seed"] = 300
narrow = session.prepare(narrow_request, parameters=params)
narrow_job = narrow.run()
print("任务 receipt：", narrow_job.receipt)
narrow_run = narrow_job.wait(timeout=120).result()
narrow_report = analyze_rabi(session, narrow_run)
print(narrow_report.status, narrow_report.pi_amplitude, narrow_report.message)
narrow_report.figure

我的诊断：______。我下一步准备修改 ______，理由是 ______。

## 6. 独立采集验证
换一个 seed，代表教学模型中的另一批噪声；其余条件保持相同。
预期两次运行 ID 不同，候选相近（相差小于 `0.02 arb`），原始数据不是完全相同的数组。
这一步是教学检查，不是实机门质量验收；自动接受候选属于后续课程。

In [ ]:
validation_request = request.copy()
validation_request.values["seed"] = 1200
validation = session.prepare(validation_request, parameters=params)
validation_job = validation.run()
print("任务 receipt：", validation_job.receipt)
validation_run = validation_job.wait(timeout=120).result()
validation_report = analyze_rabi(session, validation_run)
print(run.id, validation_run.id)
print(report.pi_amplitude, validation_report.pi_amplitude)
validation_report.figure

## 7. 完全没有响应
将 `no_response` 设置为 True 是明确的教学故障，不会改变设备。
预期平线、`fit_failed` 和空候选。遇到这样的实测曲线，应先检查响应和实验条件。

In [ ]:
silent_request = request.copy()
silent_request.values["seed"] = 2200
silent_request.values["no_response"] = True
silent = session.prepare(silent_request, parameters=params)
silent_job = silent.run()
print("任务 receipt：", silent_job.receipt)
silent_run = silent_job.wait(timeout=120).result()
silent_report = analyze_rabi(session, silent_run)
print(silent_report.status, silent_report.pi_amplitude, silent_report.message)
silent_report.figure

后续主题见[公共学习路线](https://github.com/scopecat-project/scopecat/blob/main/docs/getting-started/learning-path.md)。每次选择一个独立专题。关闭连接不会停止项目服务。


In [ ]:
# 完成所有修改后才取消下一行注释并单独执行
# session.close()
